# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irene501/flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

In [2]:
import os, getpass
import duckdb

# Colab: use the Secrets panel (key icon) to set HF_TOKEN, then userdata.get() picks it up.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':  f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':  f"read_parquet('{REL}/dim_content.parquet')",
    'fact_march':   f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [3]:
# Same decision point as the Week 3 data contract: features only from March 1-15,
# label only from March 16-31 -- never mixed.
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                            AS imp_h1,
        SUM(gsc_clicks)                                                 AS clicks_h1,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)               AS ctr_h1,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0)       AS avg_position_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)  AS active_days_h1
    FROM {TABLES['fact_march']}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 10
""").df()

label_frame = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31') AS imp_h2
    FROM {TABLES['fact_march']}
    GROUP BY 1, 2
""").df()

data = feature_frame.merge(label_frame, on=['client_hash_id', 'content_hash_id'], how='inner')
data['is_declining_proxy'] = (data['imp_h2'] < 0.8 * data['imp_h1']).astype(int)
data = data.dropna(subset=['ctr_h1', 'avg_position_h1']).reset_index(drop=True)

print(f"{len(data):,} content items with enough March 1-15 volume")
print(f"is_declining_proxy rate: {data['is_declining_proxy'].mean():.3f}")
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,475 content items with enough March 1-15 volume
is_declining_proxy rate: 0.296


,client_hash_id,content_hash_id,imp_h1,clicks_h1,ctr_h1,avg_position_h1,active_days_h1,imp_h2,is_declining_proxy
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,111.0,0.0,0.000000,5.222776,13,70.0,1
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,38.0,1.0,0.026316,5.218750,9,8.0,1
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,219.0,1.0,0.004566,4.004356,15,680.0,0
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,20.0,0.0,0.000000,4.625000,9,14.0,1
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,1494.0,0.0,0.000000,6.156643,14,1614.0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [4]:
import pandas as pd

feature_notes = pd.DataFrame([
    ("imp_h1",          "total GSC impressions, Mar 1-15",             "rows require imp_h1 sum >= 10 (HAVING clause) -- no further fill needed", "numeric", "yes -- sums only closed days before Mar 16"),
    ("clicks_h1",       "total GSC clicks, Mar 1-15",                  "0 is a valid, meaningful value (no clicks) -- not filled, not missing", "numeric", "yes -- same window as imp_h1"),
    ("ctr_h1",          "clicks_h1 / imp_h1",                          "undefined only if imp_h1 = 0, which the HAVING filter already excludes", "numeric", "yes -- ratio of two already-knowable h1 numbers"),
    ("avg_position_h1", "mean GSC position, Mar 1-15, positions > 0 only", "rows with all-zero positions in the window are dropped (dropna) rather than filled with 0 -- 0 means 'no data', not rank zero", "numeric", "yes -- averaged over already-closed days"),
    ("active_days_h1",  "distinct days with impressions > 0, Mar 1-15", "0 would mean no active days -- excluded by the imp_h1 >= 10 filter in practice", "numeric", "yes -- a count of past days only"),
], columns=["feature", "meaning", "missing_handling", "type", "available_before_decision_moment"])
feature_notes

,feature,meaning,missing_handling,type,available_before_decision_moment
0,imp_h1,"total GSC impressions, Mar 1-15",rows require imp_h1 sum >= 10 (HAVING clause) ...,numeric,yes -- sums only closed days before Mar 16
1,clicks_h1,"total GSC clicks, Mar 1-15","0 is a valid, meaningful value (no clicks) -- ...",numeric,yes -- same window as imp_h1
2,ctr_h1,clicks_h1 / imp_h1,"undefined only if imp_h1 = 0, which the HAVING...",numeric,yes -- ratio of two already-knowable h1 numbers
3,avg_position_h1,"mean GSC position, Mar 1-15, positions > 0 only",rows with all-zero positions in the window are...,numeric,yes -- averaged over already-closed days
4,active_days_h1,"distinct days with impressions > 0, Mar 1-15",0 would mean no active days -- excluded by the...,numeric,yes -- a count of past days only


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ["imp_h1", "clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1"]
leaky_features = honest_features + ["imp_h2"]  # the trap: the label's own input, smuggled in

def quick_auc(cols):
    X = data[cols]
    y = data["is_declining_proxy"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

honest_auc = quick_auc(honest_features)
leaky_auc = quick_auc(leaky_features)
print(f"honest ROC-AUC (h1 features only):        {honest_auc:.3f}")
print(f"leaky ROC-AUC (h1 features + imp_h2):      {leaky_auc:.3f}")
print()
# TODO once run: leaky_auc should print noticeably higher than honest_auc.
# If it does, that confirms the leak; if it doesn't, look harder before trusting either number --
# a leak that doesn't move the score is a sign the test itself needs fixing, not that you're safe.
print("Gap (leaky - honest):", round(leaky_auc - honest_auc, 3))

honest ROC-AUC (h1 features only):        0.591
leaky ROC-AUC (h1 features + imp_h2):      1.000

Gap (leaky - honest): 0.409


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [6]:
excluded = pd.DataFrame([
    ("imp_h2 / anything from Mar 16-31",           "the label's own input -- shown above to leak"),
    ("client_hash_id, content_hash_id",             "pseudonyms -- grouping/joining and the client-holdout split only, never a model feature"),
    ("content_created_at (dim_content)",            "excluded in the Week 3 contract's 5-feature limit; content-age effects are a fine follow-up, out of scope here for consistency with that contract"),
    ("health_score / priority_score / action_type / refresh_tier", "FlyRank product decisions, not shipped in this release, and circular if ever rebuilt and reused as a feature"),
    ("report_date",                                  "defines the h1/h2 split itself -- context, never a model input"),
], columns=["field", "why excluded"])
excluded

,field,why excluded
0,imp_h2 / anything from Mar 16-31,the label's own input -- shown above to leak
1,"client_hash_id, content_hash_id",pseudonyms -- grouping/joining and the client-...
2,content_created_at (dim_content),excluded in the Week 3 contract's 5-feature li...
3,health_score / priority_score / action_type / ...,"FlyRank product decisions, not shipped in this..."
4,report_date,"defines the h1/h2 split itself -- context, nev..."


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.